[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_07_research_capstone/ch33_reproducible_experiments_ablation/01_walkthrough.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 33 Walkthrough: Designing Rigorous Experiments — Config-Driven Runs, Multi-Seed Benchmarks & Ablation Studies

**Book Reference:** Chapter 33 (Opening Part VII: Research Capstone), Sections 33.1 – 33.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every YAML config and JSONL ledger runs cleanly inside `tempfile.TemporaryDirectory()` with zero leftover files.

In [ ]:
# @title Step 0: Click Play (▶) to Load Packages & MCH Survey Data
!pip install -q sharukh-tutor pyyaml
import json
from pathlib import Path
import random
import tempfile
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import torch
import yaml
import sharukh_tutor as tutor

mch_raw = tutor.load_data("mch_survey_sample", as_pandas=True)
mch = mch_raw.copy()
mch["residence"] = (
    mch["residence"].astype(str).str.strip().str.title()
)
mch["anc_visits"] = mch["anc_visits"].replace(999, np.nan)
mch["is_urban"] = (mch["residence"] == "Urban").astype(int)
wealth_map = {
    "Poorest": 1, "Poorer": 2, "Middle": 3,
    "Richer": 4, "Richest": 5,
}
mch["wealth_rank"] = (
    mch["wealth_quintile"].astype(str).str.strip()
    .str.title().map(wealth_map)
)
feat_cols = [
    "maternal_age_years", "is_urban", "anc_visits",
    "wealth_rank", "maternal_hb_g_dl",
]
clean = (
    mch[feat_cols + ["child_stunted", "cluster_id"]]
    .dropna()
    .reset_index(drop=True)
)
X = clean[feat_cols]
y = clean["child_stunted"].astype(int)
clusters = clean["cluster_id"]
print("Clean MCH cohort:", clean.shape,
      "| stunted:", int(y.sum()),
      "| clusters:", clusters.nunique())


## Section 33.3.1 — Multi-Library Deterministic Seed Control

In [ ]:
def set_all_seeds(seed=42):
    """Lock Python, NumPy, and PyTorch RNGs and verify scalar draws."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    return {
        "seed": int(seed),
        "py_draw": round(float(random.random()), 6),
        "np_draw": round(float(np.random.rand()), 6),
        "torch_draw": round(float(torch.rand(1).item()), 6),
    }

run1 = set_all_seeds(42)
run2 = set_all_seeds(42)
run3 = set_all_seeds(101)
print("Run 1 (seed=42): ", run1)
print("Run 2 (seed=42): ", run2)
print("Bit-exact match?:", run1 == run2)
print("Run 3 (seed=101):", run3)


## Section 33.3.2 — Config-Driven Runs With YAML & Append-Only JSONL Ledgers

In [ ]:
def load_and_validate_config(yaml_path):
    """Load YAML config and validate required experiment fields."""
    with open(yaml_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    if not isinstance(cfg, dict):
        raise ValueError("Config YAML must be a mapping.")
    required = {"experiment_name", "seeds", "model_type", "cv_splits"}
    missing = required - set(cfg.keys())
    if missing:
        raise ValueError(
            f"Missing required config keys: {sorted(missing)}"
        )
    seeds = cfg["seeds"]
    valid_seeds = (
        isinstance(seeds, list)
        and len(seeds) > 0
        and all(
            isinstance(s, int) and not isinstance(s, bool)
            for s in seeds
        )
    )
    if not valid_seeds:
        raise ValueError("seeds must be a non-empty list of ints.")
    cv = cfg["cv_splits"]
    if not isinstance(cv, int) or isinstance(cv, bool) or cv < 2:
        raise ValueError("cv_splits must be an integer >= 2.")
    return cfg

def append_experiment_ledger(ledger_path, run_record):
    """Append one UTF-8 JSON line and return total non-empty lines."""
    path = Path(ledger_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(run_record, sort_keys=True) + "\n")
    with open(path, "r", encoding="utf-8") as f:
        return int(sum(1 for line in f if line.strip()))

sample_cfg = {
    "experiment_name": "mch_stunting_lr_v1",
    "seeds": [7, 21, 42, 84, 105],
    "model_type": "logistic_regression",
    "cv_splits": 5,
}
with tempfile.TemporaryDirectory() as tmpdir:
    cfg_file = Path(tmpdir) / "exp_lr.yaml"
    ledger_file = Path(tmpdir) / "experiments.jsonl"
    cfg_file.write_text(
        yaml.safe_dump(sample_cfg, sort_keys=False), encoding="utf-8"
    )
    cfg = load_and_validate_config(cfg_file)
    n1 = append_experiment_ledger(
        ledger_file, {"exp": cfg["experiment_name"], "auc": 0.6942}
    )
    n2 = append_experiment_ledger(
        ledger_file, {"exp": "mch_stunting_rf_v1", "auc": 0.6998}
    )
    last_ln = ledger_file.read_text(encoding="utf-8").splitlines()[-1]
    print("Validated:", cfg["experiment_name"], "| seeds:", cfg["seeds"])
    print("Ledger entries after 2 runs:", n2)
    print("Last JSONL line:", last_ln)


## Section 33.3.3 — Multi-Seed Cross-Validation Reporting

In [ ]:
def multi_seed_cv_benchmark(
    X_in, y_in, model_factory, seeds, cv_splits=5
):
    """Benchmark a model factory across multiple CV split seeds."""
    raw = []
    for s in seeds:
        skf = StratifiedKFold(
            n_splits=cv_splits, shuffle=True, random_state=s
        )
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model_factory()),
        ])
        sc = cross_val_score(pipe, X_in, y_in, cv=skf, scoring="roc_auc")
        raw.append(float(sc.mean()))
    return {
        "n_seeds": int(len(seeds)),
        "mean_auc": round(float(np.mean(raw)), 4),
        "std_auc": round(float(np.std(raw)), 4),
        "min_auc": round(float(np.min(raw)), 4),
        "max_auc": round(float(np.max(raw)), 4),
        "seed_scores": [round(v, 4) for v in raw],
    }

seeds5 = [7, 21, 42, 84, 105]
lr_bench = multi_seed_cv_benchmark(
    X, y, lambda: LogisticRegression(max_iter=500), seeds5
)
rf_bench = multi_seed_cv_benchmark(
    X, y,
    lambda: RandomForestClassifier(
        n_estimators=80, max_depth=5, random_state=42
    ),
    seeds5,
)
print("LR 5-seed:", lr_bench)
print("RF 5-seed:", rf_bench)


## Section 33.3.4 — Bootstrap CIs, Paired Bootstrap & Exact McNemar Comparison

In [ ]:
def bootstrap_metric_ci(
    y_true, y_score, metric_fn, n_boot=300, alpha=0.05, seed=42
):
    """Compute non-parametric percentile bootstrap CI & SE."""
    yt, ys = np.asarray(y_true), np.asarray(y_score)
    point = float(metric_fn(yt, ys))
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(yt), size=len(yt))
        if len(np.unique(yt[idx])) < 2:
            continue
        boots.append(float(metric_fn(yt[idx], ys[idx])))
    lo, hi = np.percentile(
        boots, [100 * alpha / 2, 100 * (1 - alpha / 2)]
    )
    return {
        "point": round(point, 4),
        "ci_low": round(float(lo), 4),
        "ci_high": round(float(hi), 4),
        "se": round(float(np.std(boots, ddof=1)), 4),
    }

def paired_bootstrap_test(
    y_true, score_a, score_b, metric_fn, n_boot=400, seed=42
):
    """Run paired bootstrap test of metric(B) - metric(A)."""
    yt, sa, sb = (
        np.asarray(y_true), np.asarray(score_a), np.asarray(score_b)
    )
    delta_pt = float(metric_fn(yt, sb) - metric_fn(yt, sa))
    rng = np.random.default_rng(seed)
    deltas = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(yt), size=len(yt))
        if len(np.unique(yt[idx])) < 2:
            continue
        d = float(
            metric_fn(yt[idx], sb[idx]) - metric_fn(yt[idx], sa[idx])
        )
        deltas.append(d)
    arr = np.asarray(deltas)
    lo, hi = np.percentile(arr, [2.5, 97.5])
    p_raw = 2.0 * min(float(np.mean(arr <= 0)), float(np.mean(arr >= 0)))
    p_val = float(np.clip(p_raw, 0.0, 1.0))
    return {
        "delta_point": round(delta_pt, 4),
        "ci_low": round(float(lo), 4),
        "ci_high": round(float(hi), 4),
        "p_value_two_sided": round(p_val, 4),
    }

def mcnemar_exact_comparison(y_true, pred_a, pred_b):
    """Run exact binomial McNemar test on paired binary predictions."""
    yt, pa, pb = (
        np.asarray(y_true), np.asarray(pred_a), np.asarray(pred_b)
    )
    a_ok, b_ok = (pa == yt), (pb == yt)
    b = int(np.sum(a_ok & ~b_ok))
    c = int(np.sum(~a_ok & b_ok))
    p_val = (
        1.0 if (b + c) == 0
        else float(binomtest(min(b, c), b + c, 0.5).pvalue)
    )
    return {
        "b_only_a_correct": b,
        "c_only_b_correct": c,
        "p_value": round(p_val, 4),
        "significant_05": bool(p_val < 0.05),
    }

skf_oof = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=500)),
])
pipe_rf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(
        n_estimators=80, max_depth=5, random_state=42
    )),
])
prob_lr = cross_val_predict(
    pipe_lr, X, y, cv=skf_oof, method="predict_proba"
)[:, 1]
prob_rf = cross_val_predict(
    pipe_rf, X, y, cv=skf_oof, method="predict_proba"
)[:, 1]
print("LR 95% CI:", bootstrap_metric_ci(y, prob_lr, roc_auc_score))
print("RF 95% CI:", bootstrap_metric_ci(y, prob_rf, roc_auc_score))
print(
    "Paired Bootstrap (LR - RF):",
    paired_bootstrap_test(y, prob_rf, prob_lr, roc_auc_score),
)
pred_rf = (prob_rf >= 0.25).astype(int)
pred_lr = (prob_lr >= 0.25).astype(int)
print(
    "McNemar Exact (RF vs LR @ t=0.25):",
    mcnemar_exact_comparison(y, pred_rf, pred_lr),
)


## Section 33.3.5 — Systematic Feature-Group Ablation Study on `mch_survey_sample`

In [ ]:
def run_feature_ablation(
    X_df, y_in, feature_groups, model_factory, cv_splits=5, seed=42
):
    """Run leave-one-group-out feature ablation under Stratified CV."""
    skf = StratifiedKFold(
        n_splits=cv_splits, shuffle=True, random_state=seed
    )
    pipe_full = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model_factory()),
    ])
    full_sc = cross_val_score(
        pipe_full, X_df, y_in, cv=skf, scoring="roc_auc"
    )
    full_auc = float(full_sc.mean())
    rows = [{
        "ablation": "full_model",
        "mean_auc": round(full_auc, 4),
        "std_auc": round(float(full_sc.std()), 4),
        "delta_vs_full": 0.0,
    }]
    for gname, cols in feature_groups.items():
        pipe_g = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model_factory()),
        ])
        sc = cross_val_score(
            pipe_g, X_df.drop(columns=cols), y_in,
            cv=skf, scoring="roc_auc",
        )
        m = float(sc.mean())
        rows.append({
            "ablation": f"drop_{gname}",
            "mean_auc": round(m, 4),
            "std_auc": round(float(sc.std()), 4),
            "delta_vs_full": round(m - full_auc, 4),
        })
    return pd.DataFrame(rows)

feature_groups = {
    "demographics": ["maternal_age_years", "is_urban"],
    "care_access": ["anc_visits"],
    "socioeconomic": ["wealth_rank"],
    "nutrition": ["maternal_hb_g_dl"],
}
ablation_df = run_feature_ablation(
    X, y, feature_groups,
    lambda: LogisticRegression(max_iter=500),
    cv_splits=5, seed=42,
)
print(ablation_df.to_string(index=False))


## Section 33.3.6 — Break It On Purpose! (Experiment Harness Error Anatomy)

In [ ]:
# Bug #1: ValueError — YAML config missing required keys
with tempfile.TemporaryDirectory() as tmpdir:
    bad_cfg = Path(tmpdir) / "missing_keys.yaml"
    bad_cfg.write_text(
        "experiment_name: incomplete_run\nseeds: [42]\n",
        encoding="utf-8",
    )
    load_and_validate_config(bad_cfg)


In [ ]:
# Bug #2: ValueError — empty seeds list / cv_splits < 2
with tempfile.TemporaryDirectory() as tmpdir:
    bad_seeds = Path(tmpdir) / "empty_seeds.yaml"
    bad_seeds.write_text(
        "experiment_name: bad\nseeds: []\n"
        "model_type: lr\ncv_splits: 1\n",
        encoding="utf-8",
    )
    load_and_validate_config(bad_seeds)


In [ ]:
# Bug #3: KeyError — dropping a non-existent feature during ablation
bad_groups = {"typo_nutrition": ["maternal_bmi"]}
run_feature_ablation(
    X, y, bad_groups, lambda: LogisticRegression(max_iter=500)
)


## Section 33.4 — Stage 4 Blueprint: Config-Driven Multi-Seed Study & Ledger Audit

In [ ]:
def run_reproducible_study(raw_df, config_dict):
    """Execute config-driven benchmark, paired test & ablation."""
    df = raw_df.copy()
    df["residence"] = (
        df["residence"].astype(str).str.strip().str.title()
    )
    df["anc_visits"] = df["anc_visits"].replace(999, np.nan)
    df["is_urban"] = (df["residence"] == "Urban").astype(int)
    w_map = {
        "Poorest": 1, "Poorer": 2, "Middle": 3,
        "Richer": 4, "Richest": 5,
    }
    df["wealth_rank"] = (
        df["wealth_quintile"].astype(str).str.strip()
        .str.title().map(w_map)
    )
    f_groups = {
        "demographics": ["maternal_age_years", "is_urban"],
        "care_access": ["anc_visits"],
        "socioeconomic": ["wealth_rank"],
        "nutrition": ["maternal_hb_g_dl"],
    }
    cols = [c for g in f_groups.values() for c in g]
    cln = df[cols + ["child_stunted"]].dropna().reset_index(drop=True)
    X_c, y_c = cln[cols], cln["child_stunted"].astype(int)

    with tempfile.TemporaryDirectory() as tmpdir:
        cfg_path = Path(tmpdir) / "study_config.yaml"
        led_path = Path(tmpdir) / "study_ledger.jsonl"
        cfg_path.write_text(
            yaml.safe_dump(config_dict, sort_keys=False),
            encoding="utf-8",
        )
        cfg = load_and_validate_config(cfg_path)
        s0 = cfg["seeds"][0]
        set_all_seeds(s0)

        lr_res = multi_seed_cv_benchmark(
            X_c, y_c, lambda: LogisticRegression(max_iter=500),
            cfg["seeds"], cv_splits=cfg["cv_splits"],
        )
        rf_res = multi_seed_cv_benchmark(
            X_c, y_c,
            lambda: RandomForestClassifier(
                n_estimators=80, max_depth=5, random_state=s0
            ),
            cfg["seeds"], cv_splits=cfg["cv_splits"],
        )
        cv_oof = StratifiedKFold(
            n_splits=cfg["cv_splits"], shuffle=True, random_state=s0
        )
        p_lr = cross_val_predict(
            Pipeline([
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=500)),
            ]),
            X_c, y_c, cv=cv_oof, method="predict_proba",
        )[:, 1]
        p_rf = cross_val_predict(
            Pipeline([
                ("scaler", StandardScaler()),
                ("model", RandomForestClassifier(
                    n_estimators=80, max_depth=5, random_state=s0
                )),
            ]),
            X_c, y_c, cv=cv_oof, method="predict_proba",
        )[:, 1]
        pb = paired_bootstrap_test(
            y_c, p_rf, p_lr, roc_auc_score, seed=s0
        )
        ab = run_feature_ablation(
            X_c, y_c, f_groups,
            lambda: LogisticRegression(max_iter=500),
            cv_splits=cfg["cv_splits"], seed=s0,
        )
        drops = ab[ab["ablation"] != "full_model"]
        top_grp = str(
            drops.loc[drops["delta_vs_full"].idxmin(), "ablation"]
        ).removeprefix("drop_")
        rec = {
            "experiment_name": cfg["experiment_name"],
            "lr_mean_auc": lr_res["mean_auc"],
            "rf_mean_auc": rf_res["mean_auc"],
            "paired_delta": pb["delta_point"],
            "paired_p_value": pb["p_value_two_sided"],
            "most_critical_group": top_grp,
        }
        rec["ledger_entries"] = append_experiment_ledger(led_path, rec)
        return rec

study_cfg = {
    "experiment_name": "ch33_mch_stunting_audit",
    "seeds": [7, 21, 42, 84, 105],
    "model_type": "logistic_vs_rf",
    "cv_splits": 5,
}
print(run_reproducible_study(mch_raw, study_cfg))
